In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import glob, os

# load and stack all six seasons
files = sorted(glob.glob("data/E2_*.csv"))
frames = []
for f in files:
    season = os.path.basename(f).replace("E2_", "").replace(".csv", "")
    s = pd.read_csv(f)[["Date","HomeTeam","AwayTeam","FTHG","FTAG"]].copy()
    s["Season"] = season
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True, errors="coerce")
    frames.append(s)
allmatches = pd.concat(frames, ignore_index=True)

# per-season backtest with NO shrinkage (baseline), returns pooled predictions
def backtest_season(season_df, k=0.0):
    s = season_df.sort_values("Date").reset_index(drop=True)
    split = int(len(s) * 0.70)
    tr, te = s.iloc[:split], s.iloc[split:]

    league_home = tr["FTHG"].mean()
    league_away = tr["FTAG"].mean()

    # raw per-team sums and counts (so we can shrink later)
    hs = tr.groupby("HomeTeam").agg(goals=("FTHG","sum"), conc=("FTAG","sum"), n=("FTHG","size"))
    as_ = tr.groupby("AwayTeam").agg(goals=("FTAG","sum"), conc=("FTHG","sum"), n=("FTAG","size"))

    # shrink each team's rate toward the league average, weighted by games played
    def shrink(total, n, league):
        return (total + k*league) / (n + k)

    h_scored  = {t: shrink(r.goals, r.n, league_home) for t,r in hs.iterrows()}
    h_conceded= {t: shrink(r.conc,  r.n, league_away) for t,r in hs.iterrows()}
    a_scored  = {t: shrink(r.goals, r.n, league_away) for t,r in as_.iterrows()}
    a_conceded= {t: shrink(r.conc,  r.n, league_home) for t,r in as_.iterrows()}

    def predict(ht, at):
        he = h_scored[ht] * a_conceded[at] / league_home
        ae = a_scored[at] * h_conceded[ht] / league_away
        hw,dr,aw = 0.0,0.0,0.0
        for hg in range(10):
            for ag in range(10):
                p = poisson.pmf(hg,he)*poisson.pmf(ag,ae)
                if hg>ag: hw+=p
                elif hg==ag: dr+=p
                else: aw+=p
        return hw,dr,aw

    out=[]
    for _,r in te.iterrows():
        ht,at = r["HomeTeam"], r["AwayTeam"]
        if ht not in h_scored or at not in a_scored: continue
        hw,dr,aw = predict(ht,at)
        actual = "H" if r["FTHG"]>r["FTAG"] else ("A" if r["FTHG"]<r["FTAG"] else "D")
        out.append({"p_home":hw,"p_draw":dr,"p_away":aw,"actual":actual})
    return pd.DataFrame(out)

# baseline: k=0 (no shrinkage) — should reproduce session 6
pieces = [backtest_season(sdf, k=0.0) for _, sdf in allmatches.groupby("Season")]
results = pd.concat(pieces, ignore_index=True)
print(f"Pooled predictions (k=0): {len(results)}")

Pooled predictions (k=0): 996


In [3]:
def calibration_table(res):
    res = res.copy()
    res["home_won"] = (res["actual"] == "H").astype(int)
    bins = [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
    res["bucket"] = pd.cut(res["p_home"], bins)
    return res.groupby("bucket", observed=True).agg(
        n=("home_won","size"),
        avg_predicted=("p_home","mean"),
        actual_rate=("home_won","mean")
    )

calibration_table(results)

,n,avg_predicted,actual_rate
bucket,,,
"(0.0, 0.2]",113,0.145506,0.238938
"(0.2, 0.3]",159,0.250497,0.226415
"(0.3, 0.4]",181,0.349483,0.359116
"(0.4, 0.5]",191,0.450878,0.445026
"(0.5, 0.6]",159,0.545251,0.515723
"(0.6, 0.7]",127,0.645516,0.551181
"(0.7, 1.0]",66,0.762082,0.712121


In [5]:
pieces_k5 = [backtest_season(sdf, k=5.0) for _, sdf in allmatches.groupby("Season")]
results_k5 = pd.concat(pieces_k5, ignore_index=True)
print(f"Pooled predictions (k=5): {len(results_k5)}")
calibration_table(results_k5)

Pooled predictions (k=5): 996


,n,avg_predicted,actual_rate
bucket,,,
"(0.0, 0.2]",47,0.160041,0.234043
"(0.2, 0.3]",165,0.255666,0.242424
"(0.3, 0.4]",223,0.352815,0.313901
"(0.4, 0.5]",247,0.450675,0.449393
"(0.5, 0.6]",188,0.545291,0.521277
"(0.6, 0.7]",104,0.638820,0.625000
"(0.7, 1.0]",22,0.745179,0.772727
